In [1]:
import cobra
import thermo_flux
import thermo_flux.tools
import pandas as pd
import numpy as np

import os.path as path

from thermo_flux.io import load_excel as ex
from thermo_flux.io import helper_load as hl

from equilibrator_api import  Q_

from thermo_flux.solver.gurobi import variability_analysis

from cobra.flux_analysis import flux_variability_analysis

tfba predictions for a succinate condition

In [2]:
INPUT_MODEL = "datafiles/model.xlsx"
INPUT_KEGGS = "datafiles/ecoli_kegg_id.csv"
INPUT_REED = "datafiles/reed.csv"
INPUT_INCHI = "datafiles/InChIs.csv"
INPUT_GAMS = "datafiles/model_Ecoli_from-gams.xlsx"
INPUT_EXP_DATA = "datafiles/allPhysioData_formatted_forGSM_20230831.csv"
INPUT_EXP_CONC = "datafiles/allConcRange_20230912.csv"
INPUT_METABOLOMICS = "datafiles/metabolomics-Kochanowski_20230925.csv"

MODEL_NAME = "ecoli"
CONDITIONS = ["WT-Glc_I", "WT-Ace_I"]

INCLUDE_CO2 = True
INCLUDE_O2 = True
ALLOW_OTHER_EXCRETION = False
RELAX_EXP_FLUX_BOUNDS = 2.0

In [128]:
DATA='datafiles/'

In [3]:
def gen_model(name: str, model_xlsx: str, kegg: str, reed: str, inchi:str, gams: str, output_log: str, add_o2: bool, add_co2: bool, update_thermodynamics=True):
    tmodel = ex.create_thermo_model(name, model_excel=model_xlsx, keggids_csv=kegg, edit_mets={})

    # ADD HYDROXYBENZOATE TRANSPORT & EXCHANGE:
    # Define extracellular orotate:
    hbz_e = cobra.Metabolite(id="4hbz_e", compartment="e")
    hbz_e = thermo_flux.core.metabolite.ThermoMetabolite(hbz_e, model=tmodel)
    hbz_e.annotation = tmodel.metabolites.get_by_id("4hbz_c").annotation  
    tmodel.metabolites.append(hbz_e)

    # Hydroxybenzoate H+ antiporter (aaeB):
    HBZt3 = cobra.Reaction("HBZt3")
    HBZt3.lower_bound = -1000
    HBZt3.upper_bound = 1000
    HBZt3.add_metabolites({tmodel.metabolites.get_by_id("4hbz_e"): +1,
                      tmodel.metabolites.h_e: -1,
                      tmodel.metabolites.get_by_id("4hbz_c"): -1,
                      tmodel.metabolites.h_c: +1})
    HBZt3 = thermo_flux.core.reaction.ThermoReaction(HBZt3, model=tmodel)

    # Exchange:
    EX_4hbz = cobra.Reaction("EX_4hbz")
    EX_4hbz.add_metabolites({tmodel.metabolites.get_by_id("4hbz_e"): -1})
    EX_4hbz = thermo_flux.core.reaction.ThermoReaction(EX_4hbz, model=tmodel)

    tmodel.add_reactions([HBZt3, EX_4hbz])

    # ADD ISOPROPYLMALATE TRANSPORT & EXCHANGE:
    # Define extracellular orotate:
    ipm_e = cobra.Metabolite(id="3c3hmp_e", compartment="e")
    ipm_e = thermo_flux.core.metabolite.ThermoMetabolite(ipm_e, model=tmodel)
    ipm_e.annotation = tmodel.metabolites.get_by_id("3c3hmp_c").annotation  
    tmodel.metabolites.append(ipm_e)

    # Diffusion across the membrane:
    ipm_diff = cobra.Reaction("IPMex")
    ipm_diff.lower_bound = -1000
    ipm_diff.upper_bound = 1000
    ipm_diff.add_metabolites({tmodel.metabolites.get_by_id("3c3hmp_c"): -1,
                            tmodel.metabolites.get_by_id("3c3hmp_e"): +1})
    ipm_diff = thermo_flux.core.reaction.ThermoReaction(ipm_diff, model=tmodel)

    # Exchange:
    EX_3c3hmp = cobra.Reaction("EX_3c3hmp")
    EX_3c3hmp.add_metabolites({tmodel.metabolites.get_by_id("3c3hmp_e"): -1})
    EX_3c3hmp = thermo_flux.core.reaction.ThermoReaction(EX_3c3hmp, model=tmodel)

    tmodel.add_reactions([ipm_diff, EX_3c3hmp])

    # ADD OROTATE TRANSPORT AND EXCHANGE
    # Define extracellular orotate:
    oro_e = cobra.Metabolite(id="orot_e", compartment="e")
    oro_e = thermo_flux.core.metabolite.ThermoMetabolite(oro_e, model=tmodel)
    oro_e.annotation = tmodel.metabolites.orot_c.annotation  
    tmodel.metabolites.append(oro_e)

    # Diffusion across the membrane:
    oro_diff = cobra.Reaction("OROTex")
    oro_diff.lower_bound = -1000
    oro_diff.upper_bound = 1000
    oro_diff.add_metabolites({tmodel.metabolites.orot_c: -1,
                            tmodel.metabolites.orot_e: +1})
    oro_diff = thermo_flux.core.reaction.ThermoReaction(oro_diff, model=tmodel)

    # Dicarboxylate/H+ symporter (dctA):
    dcta = cobra.Reaction("DCTA")
    dcta.lower_bound = -1000
    dcta.upper_bound = 0
    dcta.add_metabolites({tmodel.metabolites.orot_e: +1,
                        tmodel.metabolites.h_e: +1,
                        tmodel.metabolites.orot_c: -1,
                        tmodel.metabolites.h_c: -1})
    dcta = thermo_flux.core.reaction.ThermoReaction(dcta, model=tmodel)

    # Exchange:
    EX_oro = cobra.Reaction("EX_oro")
    EX_oro.add_metabolites({tmodel.metabolites.orot_e: -1})
    EX_oro = thermo_flux.core.reaction.ThermoReaction(EX_oro, model=tmodel)

    tmodel.add_reactions([oro_diff, dcta, EX_oro])

    # Define thermodynamic parameters
    tmodel.pH = {"c": Q_(7.6), "e": Q_(7)} #pH
    tmodel.I = {"c": Q_(0.25,'M'), "e": Q_(0.25,'M')} #ionic stength
    tmodel.phi = {'ce':Q_(0.15,'V')} #membrane potential ‘ce’ represents the voltage between compartment ‘c’ and compartment 'e’ defined as Phic - Phie
    tmodel.pMg = {'e': Q_(3), 'c': Q_(3)}

    # Update metabolite annotations with the IDs from KEGG
    for met in tmodel.metabolites:
        met.annotation["bigg.metabolite"] = met.id[:-2]

    # Update the inchi strings of some unknown metabolites
    df = pd.read_csv(reed, header=None).set_index(0)

    unknown_mets = []
    for met in tmodel.metabolites:
        if met.id[:-2] in df.index:
            met.annotation['InChI'] = df.loc[met.id[:-2]].iloc[1]
            unknown_mets.append(met)

    for met in tmodel.metabolites:
        if ('kegg' in met.annotation):
            if (met.annotation['kegg'] in df.index):
            
                inchi = df.loc[met.annotation['kegg']].iloc[0]

                if type(inchi) is str:
                    met.annotation['InChI'] = df.loc[met.annotation['kegg']].iloc[0]
                    unknown_mets.append(met)

    # Additional data from excel spredsheet needs to be imported for some compounds with unknown structure
    sheets = ex.read_excelfile(model_xlsx)

    for met in tmodel.metabolites:
        if met.id[:-2] in sheets['Metabolites']['Unnamed: 0'].values:

            common_name = sheets['Metabolites'].loc[sheets['Metabolites']['Unnamed: 0'] == met.id[:-2]]['Unnamed: 11'].values
            protons = sheets['Metabolites'].loc[sheets['Metabolites']['Unnamed: 0'] == met.id[:-2]]['Unnamed: 9'].values
            charge =  sheets['Metabolites'].loc[sheets['Metabolites']['Unnamed: 0'] == met.id[:-2]]['Unnamed: 8'].values
            formula =  sheets['Metabolites'].loc[sheets['Metabolites']['Unnamed: 0'] == met.id[:-2]]['Unnamed: 12'].values

            met.formula = 'H'+str(protons[0])

            if len(common_name) > 0:
                met.notes['common name'] = common_name[0]
                met.charge = charge[0]
            else:
                met.notes['common name'] = ''

    # Hydrogenase reactions are missing protons
    for rxn in tmodel.reactions:
        if 'HYD' in rxn.id:
            if 'ATP' not in rxn.id:
                rxn.add_metabolites({tmodel.metabolites.h_e:2,
                                    tmodel.metabolites.h_c:-2})

    # Remove duplicate/additional reactions
    RXNS_TO_REMOVE = []
    for rxn in tmodel.reactions:
        if rxn.id.endswith("_add"):
            RXNS_TO_REMOVE.append(rxn)   
    tmodel.remove_reactions(RXNS_TO_REMOVE)

    # FDH reactions are missing protons and water is on wrong side of membrane 
    tmodel.reactions.FDH2.add_metabolites({tmodel.metabolites.h_e:2,
                                        tmodel.metabolites.h_c:-2,
                                        tmodel.metabolites.h2o_c:1,
                                        tmodel.metabolites.h2o_e:-1})
    tmodel.reactions.FDH3.add_metabolites({tmodel.metabolites.h_e:2,
                                        tmodel.metabolites.h_c:-2,
                                        tmodel.metabolites.h2o_c:1,
                                        tmodel.metabolites.h2o_e:-1})

    # Missing transporterd metabolites
    tmodel.reactions.TMAOR1e.transported_mets = {tmodel.metabolites.tmao_e: -1}
    tmodel.reactions.TMAOR2e.transported_mets = {tmodel.metabolites.tmao_e: -1}
    tmodel.reactions.GLCDe.transported_mets = {tmodel.metabolites.get_by_id('glc-D_e'): -1}

    #D MSOR1e protons are incorrect - this resets the reaction to the bigg version 
    tmodel.reactions.DMSOR1e.add_metabolites({tmodel.metabolites.h_e:-2,
                                            tmodel.metabolites.h_c:2 })
    tmodel.reactions.DMSOR2e.add_metabolites({tmodel.metabolites.h_e:-2,
                                            tmodel.metabolites.h_c:2 })

    tmodel.reactions.DMSOR1e.transported_mets = {tmodel.metabolites.dmso_e: -1}
    tmodel.reactions.DMSOR2e.transported_mets = {tmodel.metabolites.dmso_e: -1}

    tmodel.reactions.SHCHF.add_metabolites({tmodel.metabolites.scl_c:-1,
                                            tmodel.metabolites.srch_c:1})

    # Sirohydrochlorin dehydrogenase is incorrectly defined in model 
    tmodel.reactions.SHCHD2.add_metabolites({tmodel.metabolites.scl_c:2,
                                            tmodel.metabolites.srch_c:-2})

    #Update srch metabolite name
    tmodel.metabolites.srch_c.annotation = {'bigg.metabolite': 'dscl'}

    # Specific reaction that invovled chemcial transformation as part of transport
    tmodel.reactions.NMNt7.transported_mets = {tmodel.metabolites.nmn_e:-1}

    # PTS mechanism 
    for rxn in tmodel.reactions:
        if 'pts' in rxn.id:
            rxn.transported_mets = {met:stoich for met, stoich in rxn.metabolites.items() if met.compartment == 'e'}

    #in imported model any charge metabolite represents a free cation that is transported 
    for rxn in tmodel.reactions:
        if 'biomass' not in rxn.id:   # ignore biomass reaction 
            
            transported_charge = {}
            for met, stoich in rxn.metabolites.items():
                if (met in tmodel.charge_dict.values()):
                    transported_charge[met.compartment] = stoich
            
            if len(transported_charge) != 0:
                rxn.transported_charge = transported_charge

    # Load default concentration bounds from the GAMS model:
    df_conc = hl.excel_to_df(gams)["ConcLimits"]

    # Rearrange data for easier use:
    df_conc = df_conc.reset_index()
    df_conc["met"] = df_conc["dim1"] + "_"+ df_conc["dim2"]
    df_conc = df_conc.pivot_table(columns="dim3", values="Value", index="met")

    # Change upper and lower concentration bounds the values in the dataframe df_conc (taken from GAMS)
    # (values are in mM)
    for met, row in df_conc.iterrows():
        tmodel.metabolites.get_by_id(met).upper_bound = Q_(row["up"], "mM")
        tmodel.metabolites.get_by_id(met).lower_bound = Q_(row["lo"], "mM")

    # Find correct biomass reaction
    biomass_rxns = [r for r in tmodel.reactions if 'biomass' in r.id.lower()]
    for r in biomass_rxns:
        print(r.id, ":", r.reaction[:80])

    # Correct reaction is 'biomass'
    # Define the protons in biomass 
    tmodel.metabolites.biomass_c.formula = 'H74'  # estimated http://www.ncbi.nlm.nih.gov/pubmed/20506321
    tmodel.metabolites.biomass_e.formula = 'H74'

    # Assign biomass
    tmodel.metabolites.biomass_c.biomass = True
    tmodel.metabolites.biomass_e.biomass = True

    tmodel.reactions.biomass.add_metabolites({tmodel.metabolites.atp_c: -31.2622,
                                                tmodel.metabolites.h2o_c: -31.2622,
                                                tmodel.metabolites.adp_c: +31.2622,
                                                tmodel.metabolites.pi_c:  +31.2622})


    # Define biomass formation energy: dfG0(biomass) [kJ gCDW-1] = -2.692234848 fom Battley 1991
    base_dfg = (Q_(-2.692234848, "kJ/mol") * 1000) # values in J/gDW 

    tmodel.metabolites.biomass_c.dfG0 = base_dfg
    tmodel.metabolites.biomass_e.dfG0 = base_dfg

    tmodel.reactions.biomass_ce.ignore_snd = True

    if update_thermodynamics:
        for rxn in tmodel.reactions:
            thermo_flux.tools.drg_tools.reaction_balance(rxn, balance_charge=True, balance_mg=False)

        for met in tmodel.metabolites:
            if met.id in ['charge_c', 'charge_m', 'charge_e']:
                met.ignore_conc = True
        tmodel.update_thermo_info(fit_unknown_dfG0=True)

    return tmodel

In [4]:
def apply_metabolome_physio_data(tmodel, condition :str, input_exp: str, input_conc: str, input_metabolomics: str, input_gams: str, relax_flux_bounds, include_CO2: bool, include_O2: bool, allow_other_excr: bool, output_log: str, open_exchanges=False, flux_limit = 100):
    "Apply metabolome and physiological data to base stoichiometric-thermodynamic ecoli model"
    df_conc = hl.excel_to_df(input_gams)["ConcLimits"]

    # Rearrange data for easier use:
    df_conc = df_conc.reset_index()
    df_conc["met"] = df_conc["dim1"] + "_"+ df_conc["dim2"]
    df_conc = df_conc.pivot_table(columns="dim3", values="Value", index="met")

    # Import experimental data:
    reg_data = pd.read_csv(input_exp)

    reg_data.set_index(["cond", "rxn"], inplace=True) 
    reg_data.head()

    # Store gas fluxes:
    reg_data_gas = reg_data.swaplevel().copy()
    reg_data_gas = reg_data_gas.loc[["EX_co2", "EX_o2"]]
    reg_data_gas = reg_data_gas.swaplevel()
    reg_data_gas

    if include_CO2 is False:
        reg_data_no_gas = reg_data.swaplevel().copy()
        reg_data_no_gas = reg_data_no_gas.drop(["EX_co2"])
        reg_data_no_gas = reg_data_no_gas.swaplevel()
        reg_data = reg_data_no_gas
        
        
    if include_O2 is False:
        reg_data_no_gas = reg_data.swaplevel().copy()
        reg_data_no_gas = reg_data_no_gas.drop(["EX_o2"]) 
        reg_data_no_gas = reg_data_no_gas.swaplevel()
        reg_data = reg_data_no_gas

    # Set metabolite concentrations to the values in the GAMS model:   
    for met, row in df_conc.iterrows():
        tmodel.metabolites.get_by_id(met).upper_bound = Q_(row["up"], "mM")
        tmodel.metabolites.get_by_id(met).lower_bound = Q_(row["lo"], "mM")

    # Import experimental data:
    conc_data = pd.read_csv(input_conc)

    conc_data.set_index(["cond", "met"], inplace=True) 
    conc_data.head()

    available_conditions = ["WT-Glc_I", "WT-Gal_I", "WT-Fruc_I", "WT-Mann_I", "dptsG-Glc_I", "WT-Ace_I", "WT-Succ_I", "WT-Fum_I", "WT-Glyc_I", "WT-Pyr_I", "WT-GlyCAA_II"]

    # Import experimental data:
    met_data = pd.read_csv(input_metabolomics)
    met_data.set_index(["cond", "met"], inplace=True) 
    met_data.head()

    conds_with_data = list(met_data.reset_index().cond.unique())
    missing_conds = [cond for cond in available_conditions if cond not in conds_with_data]

    df_missing = pd.DataFrame({"cond": missing_conds, 
                            "met": "g6p",
                            "mean": np.nan, 
                            "sd": np.nan, }).set_index(["cond", "met"])

    # Apply metabolome data
    met_data = pd.concat([met_data, df_missing])
    met_data_all=pd.read_csv(input_metabolomics, index_col=(0,1))

    df_bounds=thermo_flux.solver.gurobi.calc_conc_bounds(tmodel,[condition],met_data_all,conc_units='mM')
    df_bounds_cond=df_bounds.loc[condition]
    for met_id,row in df_bounds_cond.iterrows():
        tmodel.metabolites.get_by_id(met_id).upper_bound=Q_(row['ub'],'M')
        tmodel.metabolites.get_by_id(met_id).lower_bound=Q_(row['lb'],'M')
    
    exchanges = [rxn.id for rxn in tmodel.exchanges]

    exchanges_to_relax = ["EX_C", "EX_h", "EX_h2o", "EX_k", "EX_nh3", "EX_pi", "EX_so4"]

    if include_CO2 is False:
        exchanges_to_relax += ["EX_co2"]
        
    if include_O2 is False:
        exchanges_to_relax += ["EX_o2"]

    if allow_other_excr is True:
        upper_bound_exchanges = 100
    else:
        upper_bound_exchanges = 0

    # Reset all flux bounds to +- flux limit (default -100 / 100):
    for rxn in tmodel.reactions:
        tmodel.reactions.get_by_id(rxn.id).lower_bound = -flux_limit
        tmodel.reactions.get_by_id(rxn.id).upper_bound = flux_limit
    
    # Add non-growth associate ATP maintenance cost:
    tmodel.reactions.ATPHYD.lower_bound = 3.15

    # Fix exchange reaction directions:
    for rxn in exchanges:
        tmodel.reactions.get_by_id(rxn).lower_bound = 0
        tmodel.reactions.get_by_id(rxn).upper_bound = upper_bound_exchanges
  
    # Relax essential exchanges:
    for rxn_rel in exchanges_to_relax:
        tmodel.reactions.get_by_id(rxn_rel).lower_bound = -flux_limit
        tmodel.reactions.get_by_id(rxn_rel).upper_bound = +flux_limit

    # Fix flux for the measured exchange reactions:
    for rxn, row in reg_data.loc[condition].iterrows():
        tmodel.reactions.get_by_id(rxn).lower_bound = -flux_limit
        tmodel.reactions.get_by_id(rxn).upper_bound = flux_limit

        if not open_exchanges:
            tmodel.reactions.get_by_id(rxn).lower_bound = row["mean"] - relax_flux_bounds * row["sd"]
            tmodel.reactions.get_by_id(rxn).upper_bound = row["mean"] + relax_flux_bounds * row["sd"]

    if condition.startswith("dptsG-Glc"):
        tmodel.reactions.GLCpts.lower_bound = 0
        tmodel.reactions.GLCpts.upper_bound = 0
        
    # Fix concentration for the measured extracellular metabolites:
    for met, row in conc_data.loc[condition].iterrows():
        tmodel.metabolites.get_by_id(met).lower_bound = Q_(1e-9, "M")
        tmodel.metabolites.get_by_id(met).upper_bound = Q_(100, "M")
        tmodel.metabolites.get_by_id(met).lower_bound = Q_(row["conc_M_min"], "M")
        tmodel.metabolites.get_by_id(met).upper_bound = Q_(row["conc_M_max"], "M")

    #Normalize reaction fluxes to native floats otherwise SBML will error out when exporting

    for met_id, row in df_bounds_cond.iterrows():
        print(f"model: {tmodel.metabolites.get_by_id(met_id).lower_bound}, {tmodel.metabolites.get_by_id(met_id).upper_bound}, metabolome: {row['lb']}, {row['ub']}")
        is_same = abs(tmodel.metabolites.get_by_id(met_id).lower_bound.m - Q_(row['lb'],'M').m) < 0.0000001 and abs(tmodel.metabolites.get_by_id(met_id).upper_bound.m - Q_(row['ub'],'M').m) < 0.0000001
        print(f"Metabolome data applied: {is_same}")
        
    for r in tmodel.reactions:
        r.lower_bound = float(r.lower_bound)
        r.upper_bound = float(r.upper_bound)

    return tmodel

In [5]:
def remove_orphan_metabolites(model):
    """
    Cleanup helper function that removes metabolites from the model that do not participate in any reaction
    """

    linked_metabolites = set()
    for rxn in model.reactions:
        for met in rxn.metabolites:
            linked_metabolites.add(met)
            
    all_metabolites = set(model.metabolites)
    orphan_metabolites = list(all_metabolites - linked_metabolites)
    
    orphan_ids = [m.id for m in orphan_metabolites]
    
    if orphan_metabolites:
        print(f"\nRemoving {len(orphan_metabolites)} orphan metabolites:")
        print(f"Orphaned IDs: {', '.join(orphan_ids)}")
        
        model.remove_metabolites(orphan_metabolites)
        
    else:
        print("\nNo orphan metabolites found. ")
        
    return len(orphan_metabolites)

In [6]:
def clean_fva_bounds(lb, ub, tol=1e-7):
    """
    Cleans numerical noise from TFVA bounds to improve solver stability (for example 5e-13 -> 0.0 )
    """
    old_lb = lb
    old_ub = ub
    if abs(lb) < tol: lb = 0.0
    if abs(ub) < tol: ub = 0.0
    
    if abs(lb - round(lb)) < tol: lb = round(lb, 8)
    if abs(ub - round(ub)) < tol: ub = round(ub, 8)

    if abs(ub - lb) < tol:
        # If abs difference is below tolerance fluxes are the same, set to average of lb/ub
        avg = (lb + ub) / 2
        lb, ub = avg, avg

    if lb > ub:
        lb = ub 
    
    print(f"Before/after: {old_lb}, {old_ub} ---> {lb}, {ub}")

    return lb, ub

In [7]:
def apply_met_tva(tmodel, met_tva_file):
    "Applies metabolite TVA results to the given model. Note that metaboltite indices need to 100% match."
    bounds_dict = dict()
    with open(met_tva_file, "r") as f:
        for line in f:
            clean_line = line.strip()
            if not clean_line:
                print(f"Skipping line {clean_line}")
                continue
            try:
                index_str, bounds_str = clean_line.split(':', 1)
                index = int(index_str.strip())

                cleaned_bounds_str = bounds_str.strip().strip('[] ')
                lower_str, upper_str = cleaned_bounds_str.split(',')

                lower = float(lower_str.strip())
                upper = float(upper_str.strip())

                linear_lower = np.exp(lower) * 1e3 # From molar conc back to millimolar
                linear_upper = np.exp(upper) * 1e3 # Same


                bounds_dict[index] = [linear_lower, linear_upper]
            except ValueError as e:
                print(f"Skipping line due to parsing error: '{line.strip()}' - Error: {e}")
            except Exception as e:
                print(f"unexpected error occurred while processing line: '{line.strip()}' - Error: {e}") 
    
    for met in tmodel.metabolites:
        met_index = tmodel.metabolites.index(met)

        if met_index not in bounds_dict:
            print(f"Skipped metabolite {met.id} as it was not found in TVA data.")
            continue

        cur_lower, cur_upper = met.lower_bound, met.upper_bound
        new_lower, new_upper = bounds_dict[met_index][0], bounds_dict[met_index][1]

        # Fix for floating point errors
        # Sometimes the upper bound can, for example, be 0.99999999 while the lower bound is 1.000000002
        # In this case both are equal, however floating point precision causes cobra to see the upper bound as smaller than the lower bound
        difference = abs(new_upper - new_lower)
        if difference < 1e-6:
            new_lower = new_upper

        met.upper_bound = Q_(new_upper, "millimolar")
        met.lower_bound = Q_(new_lower, "millimolar")

        print(f"Metabolite {met.id} - Old: {cur_lower}, {cur_upper} | New: {met.lower_bound, met.upper_bound}")

In [8]:
from cobra.flux_analysis import find_blocked_reactions

def setup_model(condition, mets_tva=None):
    "Setup the ecoli model given a condition for physiological data"
    tmodel = gen_model(MODEL_NAME, INPUT_MODEL, INPUT_KEGGS, INPUT_REED, INPUT_INCHI, INPUT_GAMS, "", True, True)
    tmodel = apply_metabolome_physio_data(tmodel, condition, INPUT_EXP_DATA, INPUT_EXP_CONC, INPUT_METABOLOMICS, INPUT_GAMS, RELAX_EXP_FLUX_BOUNDS, INCLUDE_CO2, INCLUDE_O2, allow_other_excr=False, output_log="", flux_limit=100)

    blocked = find_blocked_reactions(tmodel, open_exchanges=False, processes=1)

    tmodel.remove_reactions(blocked, remove_orphans=True)
    remove_orphan_metabolites(tmodel)

    for rxn in tmodel.reactions:
        thermo_flux.tools.drg_tools.reaction_balance(rxn, balance_charge=True, balance_mg=False)
    tmodel.update_thermo_info(fit_unknown_dfG0=True)

    #Set co2/o2 fluxes in correct directions
    tmodel.reactions.EX_co2.lower_bound = 0.0
    tmodel.reactions.EX_co2.upper_bound = 100.0
    
    tmodel.reactions.EX_o2.lower_bound = -100.0
    tmodel.reactions.EX_o2.upper_bound = 0.0

    # Apply results from metabolite TVA if present
    if mets_tva is not None:
        apply_met_tva(tmodel, mets_tva)
    
    return tmodel

### setup model

In [90]:
tmodel = gen_model(MODEL_NAME, INPUT_MODEL, INPUT_KEGGS, INPUT_REED, INPUT_INCHI, INPUT_GAMS, "", True, True)

blocked = find_blocked_reactions(tmodel, open_exchanges=False, processes=1)

tmodel.remove_reactions(blocked, remove_orphans=True)
remove_orphan_metabolites(tmodel)

for rxn in tmodel.reactions:
    thermo_flux.tools.drg_tools.reaction_balance(rxn, balance_charge=True, balance_mg=False)
tmodel.update_thermo_info(fit_unknown_dfG0=True)


['Parameters', 'Exchange reactions', 'Reactions', 'Biomass Composition', 'Transmembrane reactions', 'Metabolites', 'references', 'Transmembrane_reactions_reed', 'Transmembrane reactions_Orth', 'Transmembrane reactions old', 'Sheet3', 'log', 'subsystems']
*** Reading data from Reactions ***
unknown metabolite '2dhglcn[c]' created
unknown metabolite 'nadh[c]' created
unknown metabolite 'glcn[c]' created
unknown metabolite 'nad[c]' created
unknown metabolite 'nadph[c]' created
unknown metabolite 'nadp[c]' created
unknown metabolite '2dhguln[c]' created
unknown metabolite 'idon-L[c]' created
unknown metabolite '3hcinnm[c]' created
unknown metabolite 'o2[c]' created
unknown metabolite 'dhcinnm[c]' created
unknown metabolite 'h2o[c]' created
unknown metabolite '3hpppn[c]' created
unknown metabolite 'dhpppn[c]' created
unknown metabolite 'phthr[c]' created
unknown metabolite '4hthr[c]' created
unknown metabolite 'pi[c]' created
unknown metabolite '5dglcn[c]' created
unknown metabolite 'ru5p-D

Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`


*** Updating metabolite information ***
2dhglcn_c NOTHING DONE!
nadh_c NOTHING DONE!
glcn_c NOTHING DONE!
nad_c NOTHING DONE!
nadph_c NOTHING DONE!
nadp_c NOTHING DONE!
2dhguln_c NOTHING DONE!
idon-L_c NOTHING DONE!
3hcinnm_c NOTHING DONE!
o2_c NOTHING DONE!
dhcinnm_c NOTHING DONE!
h2o_c NOTHING DONE!
3hpppn_c NOTHING DONE!
dhpppn_c NOTHING DONE!
phthr_c NOTHING DONE!
4hthr_c NOTHING DONE!
pi_c NOTHING DONE!
5dglcn_c NOTHING DONE!
ru5p-D_c NOTHING DONE!
ara5p_c NOTHING DONE!
ACP_c NOTHING DONE!
atp_c NOTHING DONE!
ttdca_c NOTHING DONE!
amp_c NOTHING DONE!
myrsACP_c NOTHING DONE!
ppi_c NOTHING DONE!
ttdcea_c NOTHING DONE!
tdeACP_c NOTHING DONE!
hdca_c NOTHING DONE!
palmACP_c NOTHING DONE!
hdcea_c NOTHING DONE!
hdeACP_c NOTHING DONE!
ocdcea_c NOTHING DONE!
octeACP_c NOTHING DONE!
dtdp4aaddg_c NOTHING DONE!
unagamu_c NOTHING DONE!
dtdp_c NOTHING DONE!
unagamuf_c NOTHING DONE!
arbt6p_c NOTHING DONE!
g6p_c NOTHING DONE!
hqn_c NOTHING DONE!
4abut_c NOTHING DONE!
akg_c NOTHING DONE!
glu-L_c N

In [10]:
### dead-end loops removal 
### in blocked reactions we remove dead end metabolites. but some metabolite are not considered as dead ends if two different reactions allow to produce them. 
##yet it is still a dead end . So try to spot and remove these mets 
## in my stoichiometric matrix, i want to spot pair of reactions that use/produce the same pair of metabolite 
##first define columns that we are going to scan for 
list_of_reactions=[]
for fset in [m.reactions for m in tmodel.metabolites if len(m.reactions)==2 and m.compartment=='c']:
    for r in fset:
        list_of_reactions.append(r.id)
##but look at the indexes of these columns 
rxn_indexes_tolook=[tmodel.reactions.index(r) for r in set(list_of_reactions)]

In [11]:
# define stoichiometric matrix 
from itertools import combinations

S=cobra.util.array.create_stoichiometric_matrix(tmodel)
cofactors = [
    "h_c", "h2o_c",'h_e','co2tot_c','akg_c','glu-L_c','mql8_c','mqn8_c','pep_c','pyr_c','dmso_c','dms_c',
    '2dmmql8_c','2dmmq8_c','trdrd_c','trdox_c'
    ,"atp_c", "adp_c", "amp_c",'gtp_c','gdp_c','ACP_c',
    "pi_c", "ppi_c",
    "nad_c", "nadh_c",
    "nadp_c", "nadph_c",
    "coa_c", "accoa_c",
    "fadh2_c", "fad_c",'o2_c','q8_c','q8h2_c','charge_c','charge_e'
]
# is_cofactor = np.array([mid in cofactors for mid in [m.id for m in tmodel.metabolites]])
cofactor_indexes=[tmodel.metabolites.index(mid) for mid in cofactors]
### iw ant to check if for each column in S, there is another column that has two equal row
## i can take the column of S, then substract it from S, then look at the non zero rows of this column (bc we don't want to look at 0-0=0),
#  which other columns has non zero rows 

def find_reaction_pairs_with_shared_mets(S,S_cols):
    """
    S : stoichiometric matrix (m x n)
    min_shared : minimum number of shared metabolites
    """

    A = np.abs(S) ##take abs

    # A_filtered = A[~is_cofactor, :]
    pairs = []
    for col in S_cols:
        col_i=A[:,col]
        ##we have to look only at non zero AND non cofactors indexes
        non_zero=np.where(col_i!=0)[0]

        res_substract=(A-col_i[:,None])[[i for i in non_zero if i not in cofactor_indexes and tmodel.metabolites[i].compartment=='c']]
        ##then do not take the column which we just substracted
        res_substract[:,col]=1000 ## so it doesnt appear when we look at 0s
        # sum_of_cols=res_substract.sum(axis=0)
        ##count the zeros of each col
        zero_counts = np.sum(res_substract == 0, axis=0)
        matching_col=np.where(zero_counts==2)[0]
        if len(matching_col)==1:
            # print(matching_col)
            # matching_col_filtered=[midx for midx in matching_col if tmodel.reactions[midx].id not in cofactors]
            # if len(matching_col_filtered)>0:
            pairs.append([col,int(matching_col[0])])



    return pairs

rxnpairs=find_reaction_pairs_with_shared_mets(S,rxn_indexes_tolook)

In [12]:
for pair in rxnpairs:
    print(tmodel.reactions[pair[0]].id,tmodel.reactions[pair[0]].reaction,'AAAA',tmodel.reactions[pair[1]].id,tmodel.reactions[pair[1]].reaction)

##Then i manually checked whether these reactions actually corresponded to dead end loops (identify common mets and look at where these mets are in the rxn)

BETALDHy betald_c + h2o_c + nadp_c <=> glyb_c + 2.0 h_c + nadph_c AAAA BETALDHx betald_c + h2o_c + nad_c <=> glyb_c + 2.0 h_c + nadh_c
GLYOX h2o_c + lgt-S_c <=> gthrd_c + 1.115199703249207 h_c + lac-D_c AAAA LGTHL gthrd_c + 0.11519970324920692 h_c + mthgxl_c <=> lgt-S_c
RMPA rml1p_c <=> dhap_c + lald-L_c AAAA FCLPA fc1p_c <=> dhap_c + lald-L_c
PYDAMK atp_c + pydam_c <=> adp_c + 1.2550484116921332 h_c + pyam5p_c AAAA HYPOE h2o_c + 0.2550484116921332 h_c + pyam5p_c <=> pi_c + pydam_c
UDPG4E udpg_c <=> udpgal_c AAAA UGLT gal1p_c + udpg_c <=> g1p_c + udpgal_c
HYPOE h2o_c + 0.2550484116921332 h_c + pyam5p_c <=> pi_c + pydam_c AAAA PYDAMK atp_c + pydam_c <=> adp_c + 1.2550484116921332 h_c + pyam5p_c
BSORx btnso_c + h_c + nadh_c <=> btn_c + h2o_c + nad_c AAAA BSORy btnso_c + h_c + nadph_c <=> btn_c + h2o_c + nadp_c
ANS chor_c + gln-L_c <=> anth_c + glu-L_c + h_c + pyr_c AAAA ADCS chor_c + gln-L_c <=> 4adcho_c + glu-L_c + 0.20546014533604762 h_c
GALS3 h2o_c + melib_c <=> gal_c + glc-D_c AAAA L

In [101]:
### we found some additional reactions we could remove 
#TMAOR2, HYPOE, PYDAMK, BSORx,BSORy,BETALDHy and x, PYDXPP and PYDXK, 2DGLCNry and x
rxn_toremove =['TMAOR1','TMAOR2','HYPOE','PYDAMK','BSORx','BSORy','BETALDHy','BETALDHx','PYDXPP','PDXPP','PYDXK','PYDXNK','2DGLCNRy','2DGLCNRx']

In [102]:
tmodel.remove_reactions(rxn_toremove, remove_orphans=True)
remove_orphan_metabolites(tmodel)

tmodel.update_thermo_info()


No orphan metabolites found. 
Identifying compounds...


[████████████████████████████████████████] 604/604 orot_e                                                                                                                              

Estimating dfG0'...
[████████████████████████████████████████] 604/604 orot_e                                                                                                                                                                                                                                                                   

Estimating drG0'...
[████████████████████████████████████████] 897/897 EX_oro                                                                                                                                                                                                                                                                                                                                                                                                      



In [105]:

#Set co2/o2 fluxes in correct directions
tmodel.reactions.EX_co2.lower_bound = 0.0
tmodel.reactions.EX_co2.upper_bound = 100.0

tmodel.reactions.EX_o2.lower_bound = -100.0
tmodel.reactions.EX_o2.upper_bound = 0.0


In [106]:
tmodel.reactions.EX_o2

Reaction identifier,EX_o2
Name,
Memory address,0x7fc21d26cc70
Stoichiometry,o2_e <-- o2_e <--
GPR,
Lower bound,-100.0
Upper bound,0.0


In [107]:
exchanges_to_relax = ["EX_C", "EX_h", "EX_h2o", "EX_k", "EX_nh3", "EX_pi", "EX_so4",'EX_co2','EX_o2']

for r in [rxn for rxn in  tmodel.boundary if rxn.id not in exchanges_to_relax]:
    # print(r.id)
    r.bounds=(0,100)

In [124]:
tmodel.reactions.EX_succ.bounds=(-15.724824,-15.724824)

tmodel.objective = tmodel.reactions.biomass_EX

tfba is too slow, reduce space first with fva and also with met conc tfva

In [125]:
for r in tmodel.reactions:
    if r.lower_bound==-1000:
        r.lower_bound=-100

    if r.upper_bound==1000:
        r.upper_bound=100
df_fva=cobra.flux_analysis.flux_variability_analysis(tmodel,fraction_of_optimum=0)

In [126]:
df_fva.loc['EX_o2']

minimum   -100.0
maximum      0.0
Name: EX_o2, dtype: float64

In [127]:
## apply fva bounds
for rid in df_fva.index:
    r=tmodel.reactions.get_by_id(rid)
    r.lower_bound=df_fva.loc[rid,'minimum']
    r.upper_bound=df_fva.loc[rid,'maximum']

In [147]:
## add metabolome data ;kochanowski has succ condition
df_metconc=pd.read_csv(f'{DATA}/metabolomics-Kochanowski_20230925.csv',index_col=(0,1))
df_metconc_succ=thermo_flux.solver.gurobi.calc_conc_bounds(tmodel,['WT-Succ_I'],df_metconc,conc_units='mM').loc['WT-Succ_I']
# df_metconc.loc['WT']
for met_id,row in df_metconc_succ.iterrows():
    tmodel.metabolites.get_by_id(met_id).upper_bound=Q_(row['ub'],'M')
    tmodel.metabolites.get_by_id(met_id).lower_bound=Q_(max(row['lb'],1e-8),'M')
    print(row)

lb    0.000088
ub    0.000160
Name: ru5p-D_c, dtype: float64
lb    0.000358
ub    0.001970
Name: atp_c, dtype: float64
lb    0.000857
ub    0.001336
Name: g6p_c, dtype: float64
lb    0.000123
ub    0.000969
Name: akg_c, dtype: float64
lb    0.030294
ub    0.040067
Name: glu-L_c, dtype: float64
lb   -0.000198
ub    0.002167
Name: gtp_c, dtype: float64
lb    0.000088
ub    0.001058
Name: adp_c, dtype: float64
lb    0.000449
ub    0.000494
Name: icit_c, dtype: float64
lb    0.002842
ub    0.003518
Name: gln-L_c, dtype: float64
lb    0.000066
ub    0.000381
Name: gdp_c, dtype: float64
lb    0.000012
ub    0.000025
Name: gmp_c, dtype: float64
lb    0.001978
ub    0.002557
Name: asp-L_c, dtype: float64
lb   -0.000353
ub    0.004000
Name: imp_c, dtype: float64
lb    0.000053
ub    0.000094
Name: r5p_c, dtype: float64
lb    0.000189
ub    0.000372
Name: asn-L_c, dtype: float64
lb    0.000581
ub    0.000917
Name: pep_c, dtype: float64
lb    0.000505
ub    0.000746
Name: dhap_c, dtype: float64
l

In [148]:
tmodel.m=None
tmodel.add_TFBA_variables(qnorm='sep_norm')



# tmodel.mvars['qm'].lb=tmodel.mvars['qm'].x
# tmodel.mvars['qm'].ub=tmodel.mvars['qm'].x

Set parameter NonConvex to value 2
Set parameter TimeLimit to value 10


In [149]:
tmodel.m.update()
tmodel.m.write('ecoli_succUR_gurm.mps')

In [150]:
# tmodel.mvars['qm'].lb=tmodel.mvars['qm'].x
# tmodel.mvars['qm'].ub=tmodel.mvars['qm'].x
# thermo_flux.solver.gurobi.model_start(tmodel,'ecoli_succUR_gurm.sol',ignore_vars=['all'],fix_vars=['qm'],fix='bound')

In [151]:
tmodel.m.params.TimeLimit=60*60*14
tmodel.m.params.FeasibilityTol=1e-4
tmodel.m.params.IntFeasTol=1e-4

tmodel.m.optimize()

Set parameter TimeLimit to value 50400
Set parameter FeasibilityTol to value 0.0001
Set parameter IntFeasTol to value 0.0001
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (linux64 - "Ubuntu 24.04.3 LTS")

CPU model: 12th Gen Intel(R) Core(TM) i7-1260P, instruction set [SSE2|AVX|AVX2]
Thread count: 16 physical cores, 16 logical processors, using up to 16 threads

Non-default parameters:
TimeLimit  50400
FeasibilityTol  0.0001
IntFeasTol  0.0001
NonConvex  2

Optimize a model with 8190 rows, 7555 columns and 217287 nonzeros (Max)
Model fingerprint: 0xe19f7776
Model has 2 linear objective coefficients
Model has 335 simple general constraints
  335 ABS
Variable types: 6658 continuous, 897 integer (897 binary)
Coefficient statistics:
  Matrix range     [3e-06, 1e+08]
  Objective range  [1e+00, 1e+00]
  Bounds range     [1e-05, 1e+08]
  RHS range        [4e-15, 1e+08]
         Consider reformulating model or setting NumericFocus parameter
         to avoid numerical issues.
Presolve remov

In [70]:
PLOTTING_DATA_FOLDER='data_forplotting/'

In [71]:
df=tmodel.solution()['v'].loc[[r.id for r in tmodel.boundary]]
df[df!=0].to_csv(f'{PLOTTING_DATA_FOLDER}/ecoli_succUR_exfluxes.csv')

In [73]:
tmodel.reactions.EX_o2

Reaction identifier,EX_o2
Name,
Memory address,0x7fc230300610
Stoichiometry,o2_e --> o2_e -->
GPR,
Lower bound,0.0
Upper bound,77.41609917419834
